In [15]:
# =============================================================================
# Phase 7 — Spark Structured Streaming Consumer
# Kafka → Real-Time Processing → HDFS (Parquet)
# Craigslist Used Vehicle Dataset
# =============================================================================

import logging
import time
import json
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    FloatType, BooleanType, TimestampType, DoubleType
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
log = logging.getLogger("VehicleStreamingConsumer")


In [16]:
# =============================================================================
# CONFIG
# =============================================================================

KAFKA_BROKER       = "localhost:9092"
KAFKA_TOPIC        = "vehicle_market_stream"
KAFKA_TOPICS_CMD   = "kafka-topics.sh"

CHECKPOINT_PATH = "hdfs://127.0.0.1:9000/user/vehicle_market/logs/checkpoints/streaming"
STREAMING_OUTPUT = "hdfs://127.0.0.1:9000/user/vehicle_market/streaming/processed"

# How long to run the streaming query (seconds)
# Set to None to run indefinitely until manually stopped
STREAM_DURATION_SEC = 120 # None runs continuously until KeyboardInterrupt

# Trigger interval — how often Spark processes a micro-batch
TRIGGER_INTERVAL    = "10 seconds"

# Minimum records expected per batch for health monitoring
MIN_RECORDS_PER_BATCH = 1

In [17]:
# =============================================================================
# 1. SPARK SESSION
# =============================================================================

spark = SparkSession.builder \
    .appName("VehicleMarket_StreamingConsumer") \
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"
    ) \
    .config("spark.sql.streaming.checkpointLocation", CHECKPOINT_PATH) \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

pipeline_start = time.time()
log.info("="*65)
log.info("  Phase 7 — Spark Structured Streaming Consumer")
log.info("  Kafka → Real-Time Processing → HDFS")
log.info("="*65)

2026-07-26 00:42:17 | INFO | =================================================================
2026-07-26 00:42:17 | INFO |   Phase 7 — Spark Structured Streaming Consumer
2026-07-26 00:42:17 | INFO |   Kafka → Real-Time Processing → HDFS
2026-07-26 00:42:17 | INFO | =================================================================


In [18]:
# =============================================================================
# 2. VALIDATE KAFKA BROKER AND TOPIC
# =============================================================================

import subprocess

log.info(f"Validating Kafka broker at: {KAFKA_BROKER}...")

def verify_kafka(broker, cmd):
    try:
        result = subprocess.run(
            [cmd, "--bootstrap-server", broker, "--list"],
            capture_output=True, text=True, timeout=10
        )
        if result.returncode == 0:
            topics = [t for t in result.stdout.strip().split("\n") if t]
            return True, topics
        return False, result.stderr.strip()
    except FileNotFoundError:
        return False, (
            f"'{cmd}' not found. Set KAFKA_TOPICS_CMD to the full path."
        )
    except Exception as e:
        return False, str(e)

broker_ok, broker_info = verify_kafka(KAFKA_BROKER, KAFKA_TOPICS_CMD)

if not broker_ok:
    raise ConnectionError(
        f"Kafka broker not reachable at {KAFKA_BROKER}.\n"
        f"Details: {broker_info}"
    )

existing_topics = broker_info if isinstance(broker_info, list) else []
log.info(f"  Broker reachable. Topics: {existing_topics}")

if KAFKA_TOPIC not in existing_topics:
    raise ValueError(
        f"Topic '{KAFKA_TOPIC}' not found in Kafka. "
        f"Run Phase 6 (Kafka Producer) before this notebook."
    )

log.info(f"  Topic '{KAFKA_TOPIC}' confirmed.")

2026-07-26 00:42:18 | INFO | Validating Kafka broker at: localhost:9092...
2026-07-26 00:42:19 | INFO |   Broker reachable. Topics: ['vehicle_market_stream']
2026-07-26 00:42:19 | INFO |   Topic 'vehicle_market_stream' confirmed.


In [19]:
# =============================================================================
# 3. DEFINE MESSAGE SCHEMA
# Must match exactly the STREAM_COLUMNS published in Phase 6
# Includes engineered features: posting_period, is_weekend, depreciation_index
# =============================================================================

log.info("Defining message schema...")

MESSAGE_SCHEMA = StructType([
    StructField("id",                  StringType(),  True),
    StructField("manufacturer",        StringType(),  True),
    StructField("model",               StringType(),  True),
    StructField("year",                IntegerType(), True),
    StructField("price",               IntegerType(), True),
    StructField("odometer",            IntegerType(), True),
    StructField("fuel",                StringType(),  True),
    StructField("transmission",        StringType(),  True),
    StructField("condition",           StringType(),  True),
    StructField("title_status",        StringType(),  True),
    StructField("drive",               StringType(),  True),
    StructField("type",                StringType(),  True),
    StructField("paint_color",         StringType(),  True),
    StructField("state",               StringType(),  True),
    StructField("lat",                 DoubleType(),  True),
    StructField("long",                DoubleType(),  True),
    StructField("posting_date",        StringType(),  True),  # parsed below
    StructField("vehicle_age",         IntegerType(), True),
    StructField("age_group",           StringType(),  True),
    StructField("price_category",      StringType(),  True),
    StructField("mileage_category",    StringType(),  True),
    StructField("price_per_mile",      DoubleType(),  True),
    StructField("posting_year",        IntegerType(), True),
    StructField("posting_month",       IntegerType(), True),
    StructField("posting_season",      StringType(),  True),
    StructField("is_luxury",           BooleanType(), True),
    StructField("is_alternative_fuel", BooleanType(), True),
    StructField("vehicle_segment",     StringType(),  True),
    StructField("clean_title",         BooleanType(), True),
    StructField("is_salvage",          BooleanType(), True),
    StructField("is_automatic",        BooleanType(), True),
    StructField("price_band",          StringType(),  True),
    StructField("posting_period",      StringType(),  True),
    StructField("is_weekend",          BooleanType(), True),
    StructField("depreciation_index",  DoubleType(),  True),
])

log.info(f"  Schema defined with {len(MESSAGE_SCHEMA.fields)} fields.")

2026-07-26 00:42:19 | INFO | Defining message schema...
2026-07-26 00:42:19 | INFO |   Schema defined with 35 fields.


In [20]:
# =============================================================================
# 4. READ KAFKA STREAM
# =============================================================================

log.info(f"Subscribing to Kafka topic: {KAFKA_TOPIC}...")

df_kafka_raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BROKER) \
    .option("subscribe", KAFKA_TOPIC) \
    .option("startingOffsets", "earliest") \
    .option("failOnDataLoss", "false") \
    .load()

log.info("  Kafka stream subscribed.")

2026-07-26 00:42:19 | INFO | Subscribing to Kafka topic: vehicle_market_stream...
2026-07-26 00:42:19 | INFO |   Kafka stream subscribed.


In [21]:
# =============================================================================
# 5. PARSE JSON MESSAGES
# =============================================================================

log.info("Parsing JSON messages...")

df_parsed = df_kafka_raw \
    .select(
        F.col("key").cast(StringType()).alias("kafka_key"),
        F.col("timestamp").alias("kafka_timestamp"),
        F.col("partition").alias("kafka_partition"),
        F.col("offset").alias("kafka_offset"),
        F.from_json(
            F.col("value").cast(StringType()),
            MESSAGE_SCHEMA
        ).alias("data")
    ) \
    .select(
        "kafka_key",
        "kafka_timestamp",
        "kafka_partition",
        "kafka_offset",
        "data.*"
    )

# Convert posting_date from string to timestamp
df_parsed = df_parsed.withColumn(
    "posting_date",
    F.to_timestamp(F.col("posting_date"))
)

log.info("  JSON parsing configured.")

2026-07-26 00:42:19 | INFO | Parsing JSON messages...
2026-07-26 00:42:19 | INFO |   JSON parsing configured.


In [22]:
# =============================================================================
# 6. STREAM VALIDATION — FIELD PRESENCE CHECK
# Verify schema fields match expected columns before processing
# =============================================================================

log.info("Validating parsed stream schema...")

expected_fields  = set(f.name for f in MESSAGE_SCHEMA.fields)
parsed_fields    = set(df_parsed.columns)
schema_fields    = {
    "kafka_key", "kafka_timestamp", "kafka_partition", "kafka_offset"
}
data_fields      = parsed_fields - schema_fields

missing_in_parse = expected_fields - data_fields

if missing_in_parse:
    raise ValueError(
        f"Parsed stream is missing fields: {missing_in_parse}. "
        f"Check that MESSAGE_SCHEMA matches Phase 6 STREAM_COLUMNS."
    )

log.info(f"  Schema validation passed — {len(data_fields)} data fields present.")

2026-07-26 00:42:19 | INFO | Validating parsed stream schema...
2026-07-26 00:42:19 | INFO |   Schema validation passed — 35 data fields present.


In [23]:
# =============================================================================
# 7. STREAM TRANSFORMATIONS
# Filter invalid records and derive lightweight streaming metrics
# =============================================================================

log.info("Configuring stream transformations...")

df_processed = df_parsed \
    .filter(
        F.col("price").isNotNull()        &
        (F.col("price") > 0)              &
        F.col("manufacturer").isNotNull() &
        (F.col("manufacturer") != "unknown") &
        F.col("state").isNotNull()        &
        F.col("year").isNotNull()
    ) \
    .withColumn(
        "processed_at",
        F.current_timestamp()
    ) \
    .withColumn(
        "price_per_mile_live",
        F.when(
            F.col("odometer").isNotNull() & (F.col("odometer") > 0),
            F.round(F.col("price") / F.col("odometer"), 4)
        ).otherwise(None)
    ) \
    .withColumn(
        "is_high_value",
        F.when(F.col("price") >= 30000, True).otherwise(False)
    )

log.info("  Transformations configured.")
log.info("  Filters: price > 0, manufacturer != unknown, state not null, year not null")
log.info("  Derived: processed_at, price_per_mile_live, is_high_value")


2026-07-26 00:42:19 | INFO | Configuring stream transformations...
2026-07-26 00:42:19 | INFO |   Transformations configured.
2026-07-26 00:42:19 | INFO |   Filters: price > 0, manufacturer != unknown, state not null, year not null
2026-07-26 00:42:19 | INFO |   Derived: processed_at, price_per_mile_live, is_high_value


In [24]:
# =============================================================================
# 8. BATCH MONITORING — foreachBatch handler
# Logs per-batch statistics and performs lightweight validation
# =============================================================================

# Accumulators for cross-batch tracking
total_records_processed = spark.sparkContext.accumulator(0)
total_batches_processed = spark.sparkContext.accumulator(0)
total_batches_empty     = spark.sparkContext.accumulator(0)

batch_stats = []
# Suitable for bounded demonstration runs.
# Long-running production jobs would persist metrics externally
# (e.g. to a database or metrics store) rather than accumulating in memory.

def process_batch(batch_df, batch_id):
    """
    foreachBatch handler.
    Called once per micro-batch with the batch DataFrame and batch ID.
    Logs statistics, validates the batch, and writes to HDFS.

    batch_df is cached at the start so that count(), agg(), and write()
    all reuse the same in-memory data without rescanning the batch.
    unpersist() is called at the end to release memory promptly.
    """
    batch_start = time.time()

    # Cache batch — avoids rescanning for each subsequent action
    batch_df = batch_df.cache()

    row_count = batch_df.count()
    total_batches_processed.add(1)

    if row_count == 0:
        total_batches_empty.add(1)
        log.info(f"  Batch {batch_id:>4} | rows: 0 | EMPTY BATCH — skipping write.")
        batch_df.unpersist()
        return

    total_records_processed.add(row_count)

    # Batch-level statistics — reuses cached batch, no rescan
    stats = batch_df.agg(
        F.round(F.avg("price"), 2).alias("avg_price"),
        F.round(F.avg("odometer"), 0).alias("avg_mileage"),
        F.round(F.avg("vehicle_age"), 1).alias("avg_age"),
        F.countDistinct("manufacturer").alias("distinct_manufacturers"),
        F.countDistinct("state").alias("distinct_states"),
        F.sum(F.col("is_luxury").cast("int")).alias("luxury_count"),
        F.sum(F.col("is_high_value").cast("int")).alias("high_value_count")
    ).collect()[0]

    batch_duration = round(time.time() - batch_start, 2)

    batch_record = {
        "batch_id"              : batch_id,
        "rows"                  : row_count,
        "avg_price"             : stats["avg_price"],
        "avg_mileage"           : stats["avg_mileage"],
        "avg_age"               : stats["avg_age"],
        "distinct_manufacturers": stats["distinct_manufacturers"],
        "distinct_states"       : stats["distinct_states"],
        "luxury_count"          : stats["luxury_count"],
        "high_value_count"      : stats["high_value_count"],
        "duration_sec"          : batch_duration,
    }
    batch_stats.append(batch_record)

    log.info(
        f"  Batch {batch_id:>4} | "
        f"rows: {row_count:>5} | "
        f"avg_price: ${stats['avg_price']:>8,.0f} | "
        f"manufacturers: {stats['distinct_manufacturers']:>3} | "
        f"states: {stats['distinct_states']:>2} | "
        f"luxury: {stats['luxury_count']:>4} | "
        f"duration: {batch_duration}s"
    )

    # Warn if batch is unexpectedly small
    if row_count < MIN_RECORDS_PER_BATCH:
        log.warning(
            f"  ⚠ Batch {batch_id} has only {row_count} record(s) "
            f"— below MIN_RECORDS_PER_BATCH ({MIN_RECORDS_PER_BATCH})."
        )

    # Write batch to HDFS — reuses cached batch, no rescan
    # Partitioned by state and posting_year for efficient downstream querying
    batch_df.write \
        .mode("append") \
        .partitionBy("state", "posting_year") \
        .parquet(STREAMING_OUTPUT)

    # Release cache after all actions are complete
    batch_df.unpersist()

In [25]:
# =============================================================================
# 9. CONFIGURE AND START STREAMING QUERY
# =============================================================================

log.info("Starting Spark Structured Streaming query...")
log.info(f"  Trigger interval   : {TRIGGER_INTERVAL}")
log.info(f"  Checkpoint path    : {CHECKPOINT_PATH}")
log.info(f"  Output path        : {STREAMING_OUTPUT}")
log.info(f"  Stream duration    : {STREAM_DURATION_SEC}s")

streaming_query = df_processed.writeStream \
    .foreachBatch(process_batch) \
    .option("checkpointLocation", CHECKPOINT_PATH) \
    .trigger(processingTime=TRIGGER_INTERVAL) \
    .start()

log.info(f"  Query ID    : {streaming_query.id}")
log.info(f"  Query Name  : {streaming_query.name}")
log.info(f"  Query Status: {streaming_query.status}")




2026-07-26 00:42:19 | INFO | Starting Spark Structured Streaming query...
2026-07-26 00:42:19 | INFO |   Trigger interval   : 10 seconds
2026-07-26 00:42:19 | INFO |   Checkpoint path    : hdfs://127.0.0.1:9000/user/vehicle_market/logs/checkpoints/streaming
2026-07-26 00:42:19 | INFO |   Output path        : hdfs://127.0.0.1:9000/user/vehicle_market/streaming/processed
2026-07-26 00:42:19 | INFO |   Stream duration    : 120s
26/07/26 00:42:19 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
2026-07-26 00:42:19 | INFO |   Query ID    : 215a8778-2dc2-4318-8967-c6180d2fd3ad
2026-07-26 00:42:19 | INFO |   Query Name  : None
2026-07-26 00:42:19 | INFO |   Query Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


In [26]:
# =============================================================================
# 10. MONITOR STREAMING QUERY
# Poll status every 15 seconds until duration expires or query stops
# =============================================================================

log.info(f"Monitoring stream for {STREAM_DURATION_SEC}s...")

stream_start   = time.time()
monitor_interval = 15     # seconds between status polls
query_exception  = None

try:
    while streaming_query.isActive:
        elapsed = time.time() - stream_start

        # Check if duration limit reached
        if STREAM_DURATION_SEC and elapsed >= STREAM_DURATION_SEC:
            log.info(
                f"  Stream duration of {STREAM_DURATION_SEC}s reached. "
                f"Stopping query..."
            )
            streaming_query.stop()
            break

        # Log current query progress
        progress = streaming_query.lastProgress
        if progress:
            input_rows   = progress.get("numInputRows", 0)
            input_rate   = round(progress.get("inputRowsPerSecond", 0), 2)
            process_rate = round(progress.get("processedRowsPerSecond", 0), 2)
            batch_dur    = progress.get("durationMs", {}).get("triggerExecution", 0)

            log.info(
                f"  [{elapsed:>5.0f}s elapsed] "
                f"input rows: {input_rows:>5} | "
                f"input rate: {input_rate:>6} rows/s | "
                f"processing rate: {process_rate:>6} rows/s | "
                f"batch duration: {batch_dur}ms"
            )

        # Surface any streaming exception immediately
        if streaming_query.exception():
            query_exception = streaming_query.exception()
            log.error(f"  ✗ Streaming exception: {query_exception}")
            streaming_query.stop()
            break

        time.sleep(monitor_interval)

except KeyboardInterrupt:
    log.info("  KeyboardInterrupt received — stopping streaming query...")
    streaming_query.stop()

# Wait for clean shutdown
streaming_query.awaitTermination(timeout=30)
log.info("  Streaming query stopped.")

2026-07-26 00:42:19 | INFO | Monitoring stream for 120s...
26/07/26 00:42:19 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
2026-07-26 00:42:19 | INFO | Python Server ready to receive messages
2026-07-26 00:42:19 | INFO | Received command c on object id p1
2026-07-26 00:42:21 | INFO |   Batch    0 | rows: 30229 | avg_price: $  15,431 | manufacturers:  39 | states: 50 | luxury: 4724 | duration: 1.17s
26/07/26 00:42:38 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 10000 milliseconds, but spent 19012 milliseconds
2026-07-26 00:42:39 | INFO | Received command c on object id p1
2026-07-26 00:42:40 | INFO |   Batch    1 | rows:  3251 | avg_price: $  15,450 | manufacturers:  39 | states: 50 | luxury:  512 | duration: 0.76s
2026-07-26 00:42:49 | INFO |   [   30s elapsed] input rows: 31242 | input rate:    0.0 rows/s | proces

In [27]:
# =============================================================================
# 11. POST-STREAM VALIDATION
# Read processed output back from HDFS and confirm record count
# =============================================================================

log.info("Running post-stream validation...")

try:
    df_output = spark.read.parquet(STREAMING_OUTPUT)
    output_record_count = df_output.count()
    output_col_count    = len(df_output.columns)

    log.info(f"  Records written to HDFS : {output_record_count}")
    log.info(f"  Output columns          : {output_col_count}")

    # Verify output contains expected key columns
    key_output_cols = [
        "manufacturer", "price", "state", "vehicle_segment",
        "price_band", "processed_at"
    ]
    missing_output_cols = [
        c for c in key_output_cols if c not in df_output.columns
    ]
    if missing_output_cols:
        log.warning(f"  ⚠ Output missing columns: {missing_output_cols}")
    else:
        log.info("  Output schema validation passed.")

    # Quick aggregate to confirm data integrity
    log.info("  Output data sample:")
    df_output.groupBy("price_band") \
        .agg(
            F.count("*").alias("records"),
            F.round(F.avg("price"), 2).alias("avg_price")
        ) \
        .orderBy(F.desc("records")) \
        .show(truncate=False)

    output_validation = "✓ PASSED"

except Exception as e:
    output_record_count = 0
    output_validation   = f"✗ FAILED: {e}"
    log.error(f"  Output validation failed: {e}")


2026-07-26 00:44:19 | INFO | Running post-stream validation...
2026-07-26 00:44:20 | INFO |   Records written to HDFS : 38712
2026-07-26 00:44:20 | INFO |   Output columns          : 42
2026-07-26 00:44:20 | INFO |   Output schema validation passed.
2026-07-26 00:44:20 | INFO |   Output data sample:


+----------+-------+---------+
|price_band|records|avg_price|
+----------+-------+---------+
|Low       |17968  |5447.1   |
|Mid       |15644  |18006.86 |
|High      |5100   |42928.61 |
+----------+-------+---------+



In [28]:
# =============================================================================
# 12. STREAMING VALIDATION REPORT
# =============================================================================

pipeline_end   = time.time()
execution_time = round(pipeline_end - pipeline_start, 2)

total_batches   = total_batches_processed.value
empty_batches   = total_batches_empty.value
active_batches  = total_batches - empty_batches
records_acc     = total_records_processed.value

# Compute cross-batch aggregates from collected stats
if batch_stats:
    all_prices   = [b["avg_price"]   for b in batch_stats if b["avg_price"]]
    all_mileages = [b["avg_mileage"] for b in batch_stats if b["avg_mileage"]]
    overall_avg_price   = round(sum(all_prices)   / len(all_prices), 2)   if all_prices   else 0
    overall_avg_mileage = round(sum(all_mileages) / len(all_mileages), 0) if all_mileages else 0
    total_luxury        = sum(b["luxury_count"]     for b in batch_stats)
    total_high_value    = sum(b["high_value_count"] for b in batch_stats)
    avg_batch_duration  = round(
        sum(b["duration_sec"] for b in batch_stats) / len(batch_stats), 2
    )
else:
    overall_avg_price   = 0
    overall_avg_mileage = 0
    total_luxury        = 0
    total_high_value    = 0
    avg_batch_duration  = 0

stream_status = "✓ SUCCESS" if not query_exception \
    else f"✗ EXCEPTION: {query_exception}"

log.info("\n" + "="*65)
log.info("    SPARK STRUCTURED STREAMING — VALIDATION REPORT")
log.info("="*65)
log.info(f"  Execution Time          : {execution_time}s")
log.info(f"  Stream Duration         : {STREAM_DURATION_SEC}s")
log.info(f"  Trigger Interval        : {TRIGGER_INTERVAL}")
log.info(f"  Kafka Broker            : {KAFKA_BROKER}")
log.info(f"  Kafka Topic             : {KAFKA_TOPIC}")
log.info(f"  Checkpoint Location     : {CHECKPOINT_PATH}")
log.info(f"  Streaming Sink          : {STREAMING_OUTPUT}")
log.info(f"  Sink Format             : Parquet (partitioned by state, posting_year)")
log.info(f"  Query ID                : {streaming_query.id}")
log.info("")
log.info(f"  Total Batches           : {total_batches}")
log.info(f"  Active Batches          : {active_batches}")
log.info(f"  Empty Batches           : {empty_batches}")
log.info(f"  Avg Batch Duration      : {avg_batch_duration}s")
log.info(f"  Records Processed       : {records_acc}")
log.info(f"  Records Written (HDFS)  : {output_record_count}")
log.info("")
log.info(f"  Overall Avg Price       : ${overall_avg_price:,.2f}")
log.info(f"  Overall Avg Mileage     : {overall_avg_mileage:,.0f}")
log.info(f"  Total Luxury Vehicles   : {total_luxury}")
log.info(f"  Total High Value (30K+) : {total_high_value}")
log.info("")
log.info(f"  Stream Status           : {stream_status}")
log.info(f"  Output Validation       : {output_validation}")
log.info("")

if batch_stats:
    log.info("  Per-Batch Summary:")
    log.info(f"  {'Batch':>6} {'Rows':>7} {'Avg Price':>12} {'Mfrs':>5} "
             f"{'States':>7} {'Luxury':>7} {'Duration':>10}")
    log.info(f"  {'-'*60}")
    for b in batch_stats:
        log.info(
            f"  {b['batch_id']:>6} {b['rows']:>7} "
            f"${b['avg_price']:>10,.0f} {b['distinct_manufacturers']:>5} "
            f"{b['distinct_states']:>7} {b['luxury_count']:>7} "
            f"{b['duration_sec']:>9}s"
        )

log.info("")
if not query_exception and output_validation.startswith("✓"):
    log.info("  ✓ Streaming Consumer complete.")
    log.info("  ✓ Processed records written to HDFS.")
    log.info("  ✓ Ready for HBase storage phase.")
else:
    log.info("  ⚠ Streaming completed with issues. Review errors above.")

log.info("="*65)

2026-07-26 00:44:20 | INFO | 
2026-07-26 00:44:20 | INFO |     SPARK STRUCTURED STREAMING — VALIDATION REPORT
2026-07-26 00:44:20 | INFO | =================================================================
2026-07-26 00:44:20 | INFO |   Execution Time          : 122.68s
2026-07-26 00:44:20 | INFO |   Stream Duration         : 120s
2026-07-26 00:44:20 | INFO |   Trigger Interval        : 10 seconds
2026-07-26 00:44:20 | INFO |   Kafka Broker            : localhost:9092
2026-07-26 00:44:20 | INFO |   Kafka Topic             : vehicle_market_stream
2026-07-26 00:44:20 | INFO |   Checkpoint Location     : hdfs://127.0.0.1:9000/user/vehicle_market/logs/checkpoints/streaming
2026-07-26 00:44:20 | INFO |   Streaming Sink          : hdfs://127.0.0.1:9000/user/vehicle_market/streaming/processed
2026-07-26 00:44:20 | INFO |   Sink Format             : Parquet (partitioned by state, posting_year)
2026-07-26 00:44:20 | INFO |   Query ID                : 215a8778-2dc2-4318-8967-c6180d2fd3ad
2026-07-